# 🦷 Dental Aligner GRPO Training — Unsloth Edition

2x faster GRPO training with 60% less VRAM using [Unsloth](https://github.com/unslothai/unsloth).

**Model:** Qwen2.5-1.5B-Instruct (4-bit QLoRA — fits on free T4)
**Time:** ~20-40 min for 20 episodes on T4

Go to **Runtime → Change runtime type → T4 GPU**

%%capture
!pip install unsloth
!pip install --no-deps trl==0.16.1
!pip install wandb fastapi uvicorn pydantic scipy numpy Pillow matplotlib requests trimesh
# openenv-core may not be on PyPI — install if available, stub if not
!pip install openenv-core 2>/dev/null || echo 'openenv-core not found, will stub'

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Clone repo
!git clone https://github.com/mehular0ra/dental-aligner-claw4s.git
%cd dental-aligner-claw4s

## 2. Start Environment Server

In [ ]:
import subprocess, time, json, math, urllib.request, os, sys

# Start server using the Colab-compatible starter (stubs openenv in subprocess)
server = subprocess.Popen(
    [sys.executable, 'start_server_colab.py'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
print('Starting server (with openenv stub)...')

SERVER = 'http://localhost:7860'
def post(endpoint, data):
    req = urllib.request.Request(f'{SERVER}{endpoint}', data=json.dumps(data).encode(), headers={'Content-Type':'application/json'})
    return json.loads(urllib.request.urlopen(req, timeout=30).read().decode(), strict=False)

# Health check with retry
for attempt in range(8):
    try:
        resp = json.loads(urllib.request.urlopen(f'{SERVER}/health', timeout=5).read())
        print(f'Server ready: {resp}')
        break
    except Exception as e:
        if attempt < 7:
            print(f'  Attempt {attempt+1}/8: waiting... ({type(e).__name__})')
            time.sleep(5)
        else:
            print(f'Server failed. Stderr:')
            print(server.stderr.read().decode()[-1000:])

In [ ]:
# Quick benchmark
!python benchmarks.py --quick

## 3. wandb Login

In [ ]:
import os

# Optional: wandb login (skip if no account — training works without it)
try:
    import wandb
    print('wandb found. Paste API key below (or press Enter to skip):')
    print('Get your key at: https://wandb.ai/authorize')

    import threading, sys

    key_result = [None]
    def get_input():
        try:
            key_result[0] = input()
        except:
            key_result[0] = ''

    t = threading.Thread(target=get_input, daemon=True)
    t.start()
    t.join(timeout=30)  # Wait 30 seconds max

    if key_result[0] and key_result[0].strip():
        os.environ['WANDB_API_KEY'] = key_result[0].strip()
        wandb.login(key=key_result[0].strip())
        USE_WANDB = True
        print('wandb logged in!')
    else:
        os.environ['WANDB_DISABLED'] = 'true'
        USE_WANDB = False
        print('Skipped wandb — training will run without logging.')
except ImportError:
    os.environ['WANDB_DISABLED'] = 'true'
    USE_WANDB = False
    print('wandb not installed — training will run without logging.')

## 4. Load Model with Unsloth (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel

# Unsloth loads 4-bit quantized model — uses ~2GB VRAM instead of 6GB
# This lets us use Qwen2.5-1.5B on a free T4 (16GB)
MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    load_in_4bit=True,
)

# Add LoRA adapters (only trains ~1% of parameters)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    use_gradient_checkpointing='unsloth',  # 60% less VRAM
)

print(f'Model loaded: {MODEL_NAME}')
print(f'Trainable params: {model.print_trainable_parameters()}')

## 5. Generate Training Prompts

In [ ]:
def make_prompt(obs, stage=0):
    progress = obs.get('per_tooth_progress', [0]*28)
    mean_p = sum(progress) / max(len(progress), 1)
    min_p = min(progress) if progress else 0
    return f"""You are an orthodontic treatment planning AI. Plan aligner stage {stage+1}/24.

STATE: Stage {stage}/24, Progress: {mean_p:.0%}, Worst: {min_p:.0%}, Violations: {obs.get('cumulative_violations',0)}

RULES: Max 0.25mm/2deg per tooth per stage. Move incisors BEFORE molars.

Output JSON with your staging plan:
{{"strategy": "brief plan", "tooth_groups": [{{"teeth": [tooth_ids], "fraction": 0.0-1.0}}]}}

fraction = interpolation toward target (0=stay, 1=jump to target). System computes exact SE(3) poses."""

N_EPISODES = 30  # More episodes since Unsloth is faster
prompts = []
difficulties = [
    {'n_perturbed_teeth': 6, 'translation_magnitude': 2.0},
    {'n_perturbed_teeth': 12, 'translation_magnitude': 3.5},
    {'n_perturbed_teeth': 18, 'translation_magnitude': 5.0},
    {'n_perturbed_teeth': 10, 'translation_magnitude': 3.0, 'jitter_probability': 0.2},
]

for i in range(N_EPISODES):
    seed = i * 7 + 42
    diff = difficulties[i % len(difficulties)]
    obs = post('/reset_stepwise', {
        'task_id': 'task_easy', 'seed': seed, 'source': 'synthetic',
        'episode_id': f'prompt_{i}', 'difficulty_params': diff
    })
    prompts.append({'prompt': make_prompt(obs), 'seed': seed, 'difficulty_params': diff})

print(f'Generated {len(prompts)} prompts across {len(difficulties)} difficulty levels')

## 6. Reward Functions

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# Format for TRL
train_dataset = [{'prompt': [{'role': 'user', 'content': p['prompt']}]} for p in prompts]

config = GRPOConfig(
    output_dir='./dental_grpo_unsloth',
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    num_generations=4,
    max_prompt_length=512,
    max_completion_length=512,
    num_train_epochs=1,
    save_steps=5,
    logging_steps=1,
    report_to='wandb' if USE_WANDB else 'none',
    run_name='dental-grpo-qwen1.5b-unsloth' if USE_WANDB else None,
    bf16=True,
    gradient_accumulation_steps=2,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_terminal, reward_occlusion],
    args=config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print(f'Config: batch={config.per_device_train_batch_size}, gens={config.num_generations}, grad_accum={config.gradient_accumulation_steps}')
print(f'Logging: {"wandb" if USE_WANDB else "none (local only)"}')
print(f'Starting training on {len(train_dataset)} episodes...')

## 7. GRPO Training with Unsloth

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# Format for TRL
train_dataset = [{'prompt': [{'role': 'user', 'content': p['prompt']}]} for p in prompts]

config = GRPOConfig(
    output_dir='./dental_grpo_unsloth',
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    num_generations=4,
    max_prompt_length=512,
    max_completion_length=512,
    num_train_epochs=1,
    save_steps=5,
    logging_steps=1,
    report_to='wandb',
    run_name='dental-grpo-qwen1.5b-unsloth',
    bf16=True,
    gradient_accumulation_steps=2,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_terminal, reward_occlusion],
    args=config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print(f'Config: batch={config.per_device_train_batch_size}, gens={config.num_generations}, grad_accum={config.gradient_accumulation_steps}')
print(f'Effective batch: {config.per_device_train_batch_size * config.gradient_accumulation_steps}')
print(f'Starting training on {len(train_dataset)} episodes...')

In [ ]:
# Train!
trainer.train()

# Print results summary
print(f'\n=== RESULTS ===')
print(f'Model: {MODEL_NAME} + LoRA r=16')
print(f'Training: {len(train_dataset)} episodes, GRPO, Unsloth 4-bit')
print(f'Terminal reward: {result.get("terminal_reward",0):.4f}')
if USE_WANDB:
    print(f'wandb: {wandb.run.get_url() if wandb.run else "N/A"}')

In [ ]:
# Save LoRA adapters (tiny — just a few MB)
model.save_pretrained('./dental_grpo_unsloth/lora')
tokenizer.save_pretrained('./dental_grpo_unsloth/lora')
print('LoRA adapters saved')

# Also save merged model for inference
model.save_pretrained_merged('./dental_grpo_unsloth/merged', tokenizer, save_method='merged_16bit')
print('Merged model saved')

In [ ]:
# Evaluate: generate plan for unseen case
FastLanguageModel.for_inference(model)

test_obs = post('/reset_stepwise', {'task_id':'task_easy','seed':999,'source':'synthetic','episode_id':'eval_unsloth'})
test_prompt = make_prompt(test_obs)

inputs = tokenizer([test_prompt], return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
completion = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print(f'Model output:\n{completion[:500]}')

result = run_episode(completion, 999, {'n_perturbed_teeth':10,'translation_magnitude':3.0})
print(f'\nTrained model reward: {result.get("terminal_reward",0):.4f}')
print(f'Occlusion: {result.get("reward_breakdown",{}).get("occlusion_composite",0):.4f}')
print(f'SLERP baseline: ~0.87')
print(f'Difference: {(result.get("terminal_reward",0)-0.87)*100:+.1f}%')

In [ ]:
# Print wandb URL for the paper
print(f'\n=== ADD TO RESEARCH NOTE ===')
print(f'wandb: {wandb.run.get_url() if wandb.run else "N/A"}')
print(f'Model: {MODEL_NAME} + LoRA r=16')
print(f'Training: {len(train_dataset)} episodes, GRPO, Unsloth 4-bit')
print(f'Terminal reward: {result.get("terminal_reward",0):.4f}')

In [ ]:
# Download LoRA adapters (small, fast)
!zip -r dental_grpo_lora.zip dental_grpo_unsloth/lora/
from google.colab import files
files.download('dental_grpo_lora.zip')